# ChaCha20

Jedná se o moderní, rychlý proudový šifrovací algoritmus (stream chipper).

Vnzikla v roce 2008 a vytrvořena byla kryptografem **Danielem J. Bernsteinem** jako vylepšená verze původní *Salsa20*.

## Důvod vzniku

Důvodem byla potřeba po moderní, extrémně rychlé bezpečné šifry, která by byla velmi rychlá výhradně třeba na mobilních zařízeních na softwarové úrovní - totiž konkurenční AES k rychlému běhu vyžaduje speciální hardwarové instrukce procesoru (v té době nebyly přítomny ve všech procesorech). 

Dnes se ChaCha20 řadí mezi nejpoužívanější algoritmy na světě, používá se například pro protokol HTTPS, TLS...

## Pravidla použití

Jak jsem zmínil, jedná se o proudovou šifru - jádrem je generátor pseudonáhodných čísel.

Na začátku si vytváři matici 4x4 o 16 slovech. Do matice se vkládají: pevné konstanty, 256b klíč, počítadlo bloků a nonce.
- Nonce: **N**umber used **once** - nemusí být tajné, jeho úkolem je prostě zajistit jedinečnost zašifrované zprávy

Matice se poté míchá 20 koly (Chacha**20**), která střídají míchaní sloupců a diagonál. Toto míchání funguje na princip ARX (Add, Rotate, XOR). Výsledkém této operace je 64 bajtový nepředvýdatelný proud datm který se dalším XORem spojí s o.t.

## Důsledky porušení kryptografického protokolu

Pro proudové šifry existují dva hlavní implementační problémy:
1. **Znovupoužití Nonce** - Pro stejný klíč nesmí nikdy být použítá stejná nonce - jinak šífra vygeneruje úplně stejný keystream 
    - Pokud by útočník zachytil dvě odlišné začifrované zprávy stejnou noncí, mohl by mezi nimi udělat XOR a získá oba původní o.t.

2. **Chybějící autentizace dat** - ChaCha20 šifruje, ale nechrání integritu
    - Útočník sice nepřečte š.t., ale může v něm změnit po cestě libovolný bit - problém například při převodu peněz..
    - V praxi se vždy používá v kombinaci s autentizačním algoritmem - standard je **ChaCha20-Poly1305**

## Prolomení
ChaCha20 je odolná proti brute force útokům - vyzkoušet všechny možné kombinace 256 bitového klíče je v dnešní době (bez kvantových počítačů) nemožné.
- Fun Fact: Jedná se totiž o možných $ 2^{256} $ různých možností klíče, tedy $ 1,15 \cdot 10^{77} $ možností (V pozorovatelném Vesmíru je zhruba $ 10^{80} $ atomů) 
    - I s miliardou nejvýkonějších počítačů, za předpokladu, že by každý z nich vyzkoušel miliardu kombinací za sekundu, by brute force útok trval **miliardy let**     
    - $ 2^{256} $ je $115\,792\,089\,237\,316\,195\,423\,570\,985\,008\,687\,907\,853\,269\,984\,665\,640\,564\,039\,457\,584\,007\,913\,129\,639\,936$ možných kombinací

Aktuální pokusy o prolomení ChaCha20 jsou založeny na matematických útocích na redukované verzi ChaCha20 (která dělá méně míchání). Pro 20 kolovou verzi neexistuje matematícký způsob, jak šifru prolomit.

Zajímavostí je, že algoritmus je i **imunní vůči** tzv. *"side-channel útokům"* - tedy nejde z procesoru vypozorovat, jak šifra funguje, jak dlouho trvá, jak je náročná...
- Útočník pro pokus rozluštit šifru často analyzují:
    - Časvání - Měření na milisekundy přesně, jak dlouho trvá zašifrovat zprávu (Šifrování z "A" trvá o 1ms déle, než z "B" a útočník dokáže rozluštit šifru)
    - Spotřeba energie - Útočník měří, kolik elektřiny procesor odebírá - zpracování 1 je náročnější než 0
    - Hluk - Útočník mikrofonem měří pískání cívky v PC, která mění frekvenci podle aktuálních výpočtů 

## Kód